In [0]:
CREATE OR REPLACE TEMPORARY VIEW patient_hcp_hco_counts AS
WITH
/* =========================================================
   MEDICAL
   - HCP NPI: COALESCE(rendering, referring)
   - HCO NPI: billing_npi only when it maps to an ORGANIZATION in kom_providers
   ========================================================= */
medical_base AS (
  SELECT DISTINCT
    me.PATIENT_ID,
    COALESCE(me.RENDERING_NPI, me.REFERRING_NPI) AS HCP_NPI,
    me.BILLING_NPI AS BILLING_NPI
  FROM com_edp_prd.com_raw.kom_medical_events me
),

medical_hco AS (
  SELECT DISTINCT
    mb.PATIENT_ID,
    mb.BILLING_NPI AS HCO_NPI
  FROM medical_base mb
  JOIN com_edp_prd.com_raw.kom_providers p
    ON mb.BILLING_NPI = p.NPI
  WHERE p.PROVIDER_TYPE = 'ORGANIZATION'
),

/* =========================================================
   PHARMACY
   - HCP NPI: prescriber_npi
   - HCO NPI: pharmacy_npi (assumed already an org identifier)
   ========================================================= */
pharmacy_base AS (
  SELECT DISTINCT
    pe.PATIENT_ID,
    pe.PRESCRIBER_NPI AS HCP_NPI,
    pe.PHARMACY_NPI   AS HCO_NPI
  FROM com_edp_prd.com_raw.kom_pharmacy_events pe
),

/* =========================================================
   COMBINED (UNION of distinct IDs)
   ========================================================= */
combined_patients AS (
  SELECT DISTINCT PATIENT_ID FROM medical_base
  UNION
  SELECT DISTINCT PATIENT_ID FROM pharmacy_base
),
combined_hcps AS (
  SELECT DISTINCT HCP_NPI FROM medical_base WHERE HCP_NPI IS NOT NULL
  UNION
  SELECT DISTINCT HCP_NPI FROM pharmacy_base WHERE HCP_NPI IS NOT NULL
),
combined_hcos AS (
  SELECT DISTINCT HCO_NPI FROM medical_hco WHERE HCO_NPI IS NOT NULL
  UNION
  SELECT DISTINCT HCO_NPI FROM pharmacy_base WHERE HCO_NPI IS NOT NULL
)

SELECT
  /* Distinct patients */
  (SELECT COUNT(DISTINCT PATIENT_ID) FROM medical_base)    AS medical_distinct_patients,
  (SELECT COUNT(DISTINCT PATIENT_ID) FROM pharmacy_base)   AS pharmacy_distinct_patients,
  (SELECT COUNT(DISTINCT PATIENT_ID) FROM combined_patients) AS combined_distinct_patients,

  /* Distinct HCP NPIs */
  (SELECT COUNT(DISTINCT HCP_NPI) FROM medical_base  WHERE HCP_NPI IS NOT NULL) AS medical_distinct_hcp_npis,
  (SELECT COUNT(DISTINCT HCP_NPI) FROM pharmacy_base WHERE HCP_NPI IS NOT NULL) AS pharmacy_distinct_hcp_npis,
  (SELECT COUNT(*) FROM combined_hcps)                                          AS combined_distinct_hcp_npis,

  /* Distinct HCO NPIs */
  (SELECT COUNT(DISTINCT HCO_NPI) FROM medical_hco   WHERE HCO_NPI IS NOT NULL) AS medical_distinct_hco_npis,
  (SELECT COUNT(DISTINCT HCO_NPI) FROM pharmacy_base WHERE HCO_NPI IS NOT NULL) AS pharmacy_distinct_hco_npis,
  (SELECT COUNT(*) FROM combined_hcos)                                          AS combined_distinct_hco_npis
;

In [0]:
select * from patient_hcp_hco_counts

In [0]:
select * from com_edp_prd.cmpa_insights_internal_schema.zip_to_territory_mapping